In [ ]:
# preprocessing package form the "toyxtoyproblem-af-classification" problem 

import sys
sys.path.append(r"C:\Users\saber\thesis_second_try\toyxtoyproblem-af-classification\ecg-preprocessing")


In [11]:
from scipy.io import loadmat

path = r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat"
data = loadmat(path)


In [18]:
from pathlib import Path

p = Path(r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat")
print("Exists:", p.exists())
print("Size (MB):", p.stat().st_size / 1e6)


Exists: True
Size (MB): 67.086024


In [19]:
p = r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat"

with open(p, "rb") as f:
    head = f.read(128)

print(head[:32])
print(head)


b'MATLAB 5.0 MAT-file, Platform: P'
b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Thu Aug 15 17:47:14 2024                                        \x00\x00\x00\x00\x00\x00\x00\x00\x00\x01IM'


In [21]:
from scipy.io import loadmat

p = r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat"
mat = loadmat(p)

print("All keys:", list(mat.keys()))


All keys: ['__header__', '__version__', '__globals__', 'aggr_data']


In [16]:
print([k for k in data.keys() if not k.startswith("__")])


['aggr_data']


In [22]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

p = r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat"
mat = loadmat(p, squeeze_me=True, struct_as_record=False)

aggr = mat["aggr_data"]
print("type(aggr):", type(aggr))

# If it's a numeric array
if isinstance(aggr, np.ndarray) and np.issubdtype(aggr.dtype, np.number):
    print("aggr.shape:", aggr.shape, "dtype:", aggr.dtype)

    x = np.squeeze(aggr)

    # If 1D: plot directly
    if x.ndim == 1:
        plt.figure(figsize=(12,4))
        plt.plot(x[:2000])
        plt.title("aggr_data (first 2000 samples)")
        plt.xlabel("sample")
        plt.ylabel("value")
        plt.show()

    # If 2D: plot first column and a heatmap overview
    elif x.ndim == 2:
        plt.figure(figsize=(12,4))
        plt.plot(x[:2000, 0])
        plt.title("aggr_data[:,0] (first 2000 samples)")
        plt.xlabel("sample")
        plt.ylabel("value")
        plt.show()

        plt.figure(figsize=(8,5))
        plt.imshow(x.T, aspect="auto")
        plt.title("aggr_data heatmap (features/leads x time)")
        plt.xlabel("time/sample")
        plt.ylabel("channel/feature")
        plt.show()

    else:
        print("Numeric but unexpected ndim:", x.ndim)

# If it's a MATLAB struct-like object
elif hasattr(aggr, "_fieldnames"):
    print("aggr_data fields:", aggr._fieldnames)

    # Print shapes/types for each field
    for f in aggr._fieldnames:
        v = getattr(aggr, f)
        if isinstance(v, np.ndarray):
            print(f"{f}: array shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"{f}: type={type(v)} value_preview={str(v)[:80]}")

    # Try to automatically pick a field to plot (largest numeric array)
    candidates = []
    for f in aggr._fieldnames:
        v = getattr(aggr, f)
        if isinstance(v, np.ndarray) and np.issubdtype(v.dtype, np.number):
            candidates.append((f, v.size))
    if candidates:
        best_field = sorted(candidates, key=lambda t: t[1], reverse=True)[0][0]
        x = np.squeeze(getattr(aggr, best_field))
        print("Auto-picked field to plot:", best_field, "shape:", x.shape)

        plt.figure(figsize=(12,4))
        if x.ndim == 1:
            plt.plot(x[:2000])
        elif x.ndim == 2:
            plt.plot(x[:2000, 0])
        else:
            print("Picked field has ndim:", x.ndim)
        plt.title(f"{best_field} (preview)")
        plt.show()
    else:
        print("No numeric arrays found inside aggr_data fields.")

else:
    # Fallback
    try:
        arr = np.array(aggr)
        print("Converted to array:", arr.shape, arr.dtype)
    except Exception as e:
        print("Could not interpret aggr_data:", e)


type(aggr): <class 'scipy.io.matlab._mio5_params.mat_struct'>
aggr_data fields: ['nose', 'mouth']
nose: type=<class 'scipy.io.matlab._mio5_params.mat_struct'> value_preview=<scipy.io.matlab._mio5_params.mat_struct object at 0x000002153CD98F40>
mouth: type=<class 'scipy.io.matlab._mio5_params.mat_struct'> value_preview=<scipy.io.matlab._mio5_params.mat_struct object at 0x000002153CD99A20>
No numeric arrays found inside aggr_data fields.


In [23]:
import numpy as np
from scipy.io import loadmat

p = r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat"
mat = loadmat(p, squeeze_me=True, struct_as_record=False)
aggr = mat["aggr_data"]

def describe(obj, name="obj", depth=0, max_depth=4):
    indent = "  " * depth
    t = type(obj).__name__
    print(f"{indent}{name}: {t}")

    # MATLAB struct
    if hasattr(obj, "_fieldnames"):
        print(f"{indent}  fields: {obj._fieldnames}")
        if depth >= max_depth:
            return
        for f in obj._fieldnames:
            v = getattr(obj, f)
            describe(v, f"{name}.{f}", depth+1, max_depth)
        return

    # numpy array
    if isinstance(obj, np.ndarray):
        print(f"{indent}  array shape={obj.shape}, dtype={obj.dtype}")
        return

    # scalar / other
    print(f"{indent}  value preview: {str(obj)[:80]}")

describe(aggr, "aggr_data")


aggr_data: mat_struct
  fields: ['nose', 'mouth']
  aggr_data.nose: mat_struct
    fields: ['eyes', 'sizepx', 'theta', 'time', 'subj', 'gender']
    aggr_data.nose.eyes: ndarray
      array shape=(1438300,), dtype=float64
    aggr_data.nose.sizepx: ndarray
      array shape=(1438300,), dtype=float64
    aggr_data.nose.theta: ndarray
      array shape=(1438300,), dtype=float64
    aggr_data.nose.time: ndarray
      array shape=(1438300,), dtype=float64
    aggr_data.nose.subj: ndarray
      array shape=(1438300,), dtype=uint8
    aggr_data.nose.gender: ndarray
      array shape=(1438300,), dtype=<U1
  aggr_data.mouth: mat_struct
    fields: ['eyes', 'sizepx', 'theta', 'time', 'subj', 'gender']
    aggr_data.mouth.eyes: ndarray
      array shape=(1364781,), dtype=float64
    aggr_data.mouth.sizepx: ndarray
      array shape=(1364781,), dtype=float64
    aggr_data.mouth.theta: ndarray
      array shape=(1364781,), dtype=float64
    aggr_data.mouth.time: ndarray
      array shape=(1364781,

In [24]:
import numpy as np
import pandas as pd
from scipy.io import loadmat

p = r"C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat"
mat = loadmat(p, squeeze_me=True, struct_as_record=False)
aggr = mat["aggr_data"]

def extract_arrays(obj, prefix=""):
    out = {}
    if hasattr(obj, "_fieldnames"):  # MATLAB struct
        for f in obj._fieldnames:
            out.update(extract_arrays(getattr(obj, f), prefix + f + "_"))
    elif isinstance(obj, np.ndarray) and np.issubdtype(obj.dtype, np.number):
        out[prefix[:-1]] = np.squeeze(obj)
    return out

arrays = extract_arrays(aggr)
print("Found arrays:", arrays.keys())


Found arrays: dict_keys(['nose_eyes', 'nose_sizepx', 'nose_theta', 'nose_time', 'nose_subj', 'mouth_eyes', 'mouth_sizepx', 'mouth_theta', 'mouth_time', 'mouth_subj'])


i have converted rest pupil_preprocess in to cvs for each feature possibly unncessary.
TODO : what to do the next try to use the preprocessing file to preprocess the rest respiration 

In [10]:
from ecgprep.read_ecg import read_ecg
import ecg_plot




PATH_TO_WFDB = 'C:\Users\saber\thesis_second_try\data\Rest\Pupil_Preprocessed\aggregated_data_combined.mat'.
ecg_sample, sample_rate, _ = read_ecg(PATH_TO_WFDB)

# ECG plot
plt.figure()

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (718522031.py, line 7)